In [24]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=gemini_api_key)

In [49]:
# query = "현재 빌보드 차트 1위 곡은?"
query = "2025년 KT 소액결제 사건의 원인은?"

In [50]:
response = llm.invoke(query)
response

AIMessage(content='2025년 KT 소액결제 사건이라는 것은 **아직 발생하지 않은 미래의 사건**입니다.\n\n따라서 현재 시점에서 그 원인에 대해 정확히 말씀드릴 수는 없습니다.\n\n다만, 과거 사례를 비추어 볼 때, 통신사 소액결제 관련 문제들은 주로 다음과 같은 원인으로 발생해왔습니다:\n\n1.  **스미싱(Smishing) 및 피싱(Phishing) 사기:** 악성 앱 설치나 가짜 결제 페이지 접속을 유도하여 소액결제를 발생시키거나 개인정보를 탈취하는 경우.\n2.  **개인정보 유출 및 도용:** 다른 경로로 유출된 개인정보를 이용해 명의를 도용하여 소액결제를 시도하는 경우.\n3.  **악성 앱 설치 유도:** 무료 쿠폰, 택배 알림 등을 위장하여 악성 앱 설치를 유도하고, 설치된 앱이 사용자 몰래 소액결제를 진행하는 경우.\n4.  **사용자 부주의 또는 보안 의식 부족:** 출처를 알 수 없는 링크 클릭, 앱 설치, 본인인증 요청에 대한 무분별한 동의 등으로 인해 피해를 입는 경우.\n\n만약 특정 시기에 발생한 소액결제 관련 문제에 대해 궁금하시다면, 해당 시점을 정확히 알려주시면 관련 정보를 찾아볼 수 있습니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--3523a9d5-d656-4077-bb56-395d360c68f3-0', usage_metadata={'input_tokens': 16, 'output_tokens': 1262, 'total_tokens': 1278, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 962}})

In [51]:
from langchain_community.tools import DuckDuckGoSearchResults 

search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search.invoke(query)

print(docs)

snippet: 1 day ago · 2025년 8월 수도권에서 발생한 KT 소액결제 해킹 사건은 불법 초소형 기지국을 통한 역대급 해킹 피해 사례입니다. 피해 현황, 해킹 원인, KT 및 정부 대응, 보상 절차와 예방법까지 …, title: KT 소액결제 해킹 사건 정리 (2025) – 피해 현황·원인·보상 절차 ..., link: https://reasonablegift.tistory.com/175;
snippet: 1 day ago · KT 대표 김영섭이 수도권 소액 결제 사고에 공식 사과하며 100% 보상을 약속했다. 불법 펨토셀이 KT 통신망에 접속해 고객 정보 유출 및 무단 결제가 ..., title: 'KT 이용자 무단 소액 결제' 사태, 원인은 가짜 기지국? [영상 ..., link: https://www.hankookilbo.com/News/Read/A2025091117380004069;
snippet: 1 day ago · 왜 KT만 털렸나...KT 무단 소액결제 해킹 사건의 의문점 KT 무단 소액 결제 사건 관련해 과학기술정통부는 지난 10일 긴급 브리핑을 열고 현재까지 피해 상황과 조사 내용 등을 …, title: 왜 KT만 털렸나...KT 무단 소액결제 해킹 사건의 의문점 - 조선일보, link: https://www.chosun.com/economy/tech_it/2025/09/11/P6IGLCLDPFFARKYDTBAPSIHFYA/;
snippet: 4 days ago · 1. 개요 [편집] 2025년 8월 말 수도권 지역에서 KT 통신사 고객의 휴대전화 무단 소액결제 피해가 발생한 사건., title: KT 소액결제 해킹 사건 - 나무위키, link: https://namu.wiki/w/KT+소액결제+해킹+사건


c:\Users\dandy\Documents\GitHub\kor-it-langchain-class\ch07\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


In [52]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = question_answering_prompt | llm

In [53]:
from langchain.memory import ChatMessageHistory

# 채팅 메시지를 저장할 메모리 객체 생성
chat_history = ChatMessageHistory() 
# 사용자 질문을 메모리에 저장
chat_history.add_user_message(query)

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs, # 검색 결과 전달
    }
)

# 생성된 답변을 메모리에 저장
chat_history.add_ai_message(answer) 

print(answer)

content='2025년 KT 소액결제 사건의 원인은 **불법 초소형 기지국(불법 펨토셀 또는 가짜 기지국)**을 통한 해킹으로 밝혀졌습니다.\n\n이 불법 기지국이 KT 통신망에 접속하여 고객 정보 유출 및 무단 결제가 발생했습니다.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []} id='run--2c9659df-301d-409a-bd76-16996d333846-0' usage_metadata={'input_tokens': 503, 'output_tokens': 598, 'total_tokens': 1101, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 524}}


### 🔎 웹 검색 도구

In [ ]:
from langchain_core.tools import tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

@tool
def get_web_search(query: str, search_period: str='w') -> str:
    """
    웹 검색을 수행하는 함수.

    Args:
        query (str): 검색어
        search_period (str): 검색 기간 (e.g., "w" for past week (default), "m" for past month, "y" for past year, "d" for past day)

    Returns:
        str: 검색 결과
    """
    wrapper = DuckDuckGoSearchAPIWrapper(
        region="kr-kr",
        time=search_period
    )

    print('\n----- WEB SEARCH -----')
    print(query)
    print(search_period)

    search = DuckDuckGoSearchResults(
        api_wrapper=wrapper,
        source="news",
        results_separator=';\n'
    )

    searched = search.invoke(query)
    
    for i, result in enumerate(searched.split(';\n')):
        print(f'{i+1}. {result}')
    
    return searched

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# 도구들을 tools 리스트에 추가
tools = [get_web_search] # 에이전트가 사용할 도구 리스트

# 에이전트 프롬프트 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    ("human", "{input}"),
    # AgentExecutor는 agent_scratchpad를 자동으로 채워 넣어,
    # 에이전트가 이전 단계의 행동과 결과를 기억하고 다음 행동을 결정할 수 있도록 함
    ("placeholder", "{agent_scratchpad}"),
])

# 에이전트 생성
agent = create_tool_calling_agent(llm, tools, prompt)

# AgentExecutor 초기화
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [56]:
from langchain_core.messages import HumanMessage

# 메시지 형식 변경 (agent_executor에 맞게)
messages = [
    HumanMessage(query),
]

# agent_executor를 사용하여 실행
response = agent_executor.invoke({"input": messages})
response



> Entering new AgentExecutor chain...
2025년 KT 소액결제 사건에 대한 정보는 현재로서는 알 수 없습니다. 해당 사건이 아직 발생하지 않았기 때문입니다.

> Finished chain.


{'input': [HumanMessage(content='2025년 KT 소액결제 사건의 원인은?', additional_kwargs={}, response_metadata={})],
 'output': '2025년 KT 소액결제 사건에 대한 정보는 현재로서는 알 수 없습니다. 해당 사건이 아직 발생하지 않았기 때문입니다.'}